In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import BaggingClassifier #also a bagging regressor available
from sklearn.ensemble import AdaBoostClassifier #also a AdaBoostRegressor
from sklearn.ensemble import RandomForestClassifier #also a random forest regressor, also voting, etc.
from sklearn.neural_network import MLPClassifier

In [2]:
crop = pd.read_csv('Unbalanced_Binary_Dataset.csv')
crop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   GrowthRate   5000 non-null   float64
 1   Consistency  5000 non-null   float64
 2   Infection    5000 non-null   float64
 3   Temperature  5000 non-null   float64
 4   Special      5000 non-null   int64  
 5   y            5000 non-null   int64  
dtypes: float64(4), int64(2)
memory usage: 234.5 KB


In [3]:
crop.head()

,GrowthRate,Consistency,Infection,Temperature,Special,y
0,0.496714,4.152481,-0.357490,1.444931,0,1
1,-0.138264,4.093172,-0.793962,1.685578,0,1
2,0.647689,1.408714,-0.856385,0.037804,1,1
3,1.523030,4.339820,-0.811448,2.484231,1,1
4,-0.234153,6.465658,0.165739,2.492330,0,1


In [4]:
#Determine beta coefficients and p-values
inputs = 'GrowthRate + Consistency + Infection + Temperature + Special'
lm = smf.ols(formula = f'y ~ {inputs}', data = crop).fit()

In [5]:
lm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.151
Model:                            OLS   Adj. R-squared:                  0.151
Method:                 Least Squares   F-statistic:                     178.3
Date:                Thu, 26 Mar 2026   Prob (F-statistic):          4.14e-175
Time:                        01:36:44   Log-Likelihood:                -2368.6
No. Observations:                5000   AIC:                             4749.
Df Residuals:                    4994   BIC:                             4788.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       0.4326      0.016     26.791      0.000       0.401       0.464
GrowthRate      0.0674      0.006     12.221      0.000       0.057       0.078
Consistency     0.0430      0.003     15.808      0.000       0.038       0.048
Infection      -0.0989      0.010    -10.407      0.000      -0.118      -0.080
Temperature     0.0302      0.003     11.439      0.000       0.025       0.035
Special         0.2002      0.012     16.562      0.000       0.176       0.224
==============================================================================
Omnibus:                      607.265   Durbin-Watson:                   2.027
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              792.086
Skew:                          -0.953   Prob(JB):                    1.00e-172
Kurtosis:                       2.593   Cond. No.                         17.8
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [6]:
#condidered dropping Special but left it in since a crop with or without special treatment could yield better results.
#It also has a very small p value much smaller than .05
#deterimine predictors(X) and outcome(y)
predictors = ['GrowthRate', 'Consistency', 'Infection', 'Temperature', 'Special']

In [7]:
y = crop['y'].astype('category')
print(y.info())

<class 'pandas.core.series.Series'>
RangeIndex: 5000 entries, 0 to 4999
Series name: y
Non-Null Count  Dtype   
--------------  -----   
5000 non-null   category
dtypes: category(1)
memory usage: 5.1 KB
None


In [8]:
crop["y"].value_counts()

,count
y,
1,3842
0,1158


In [9]:
#handle categorical variables
X = pd.get_dummies(crop[predictors], drop_first= True)

In [10]:
print(X.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   GrowthRate   5000 non-null   float64
 1   Consistency  5000 non-null   float64
 2   Infection    5000 non-null   float64
 3   Temperature  5000 non-null   float64
 4   Special      5000 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 195.4 KB
None


In [11]:
xtrain, xvalid, ytrain, yvalid = train_test_split(X, y,
                                                      test_size = 0.2, random_state=42)

print(xtrain.info())
print()
print(ytrain.info())
print()
print(xvalid.info())
print()
print(yvalid.info())

<class 'pandas.core.frame.DataFrame'>
Index: 4000 entries, 4227 to 860
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   GrowthRate   4000 non-null   float64
 1   Consistency  4000 non-null   float64
 2   Infection    4000 non-null   float64
 3   Temperature  4000 non-null   float64
 4   Special      4000 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 187.5 KB
None

<class 'pandas.core.series.Series'>
Index: 4000 entries, 4227 to 860
Series name: y
Non-Null Count  Dtype   
--------------  -----   
4000 non-null   category
dtypes: category(1)
memory usage: 35.3 KB
None

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 1501 to 1926
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   GrowthRate   1000 non-null   float64
 1   Consistency  1000 non-null   float64
 2   Infection    1000 non-null   float64
 3   Temperature  1000 n

In [12]:
overs = SMOTE()
unders = RandomUnderSampler(sampling_strategy= "majority")

xtrainos, ytrainos = overs.fit_resample(xtrain, ytrain)
xtrainus, ytrainus = unders.fit_resample(xtrain, ytrain)

print(ytrainos.value_counts())
print(ytrainus.value_counts())

y
0    3095
1    3095
Name: count, dtype: int64
y
0    905
1    905
Name: count, dtype: int64


In [13]:
#Normalizing the dataset

scale = StandardScaler()
xtrain2 = pd.DataFrame(scale.fit_transform(xtrain), columns = X.columns)
xtrainos2 = pd.DataFrame(scale.fit_transform(xtrainos), columns = X.columns)
xtrainus2 = pd.DataFrame(scale.fit_transform(xtrainus), columns = X.columns)
xvalid2 = pd.DataFrame(scale.fit_transform(xvalid), columns = X.columns)

##Logistic Reg

In [14]:
#Logistic Regression Model

#ORIGINAL
lg = LogisticRegression()
lg.fit(xtrain, ytrain)
y_pred = lg.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

#OVERSAMPLED
lg = LogisticRegression()
lg.fit(xtrainos, ytrainos)
y_pred = lg.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

#UNDERSAMPLED
lg = LogisticRegression()
lg.fit(xtrainus, ytrainus)
y_pred = lg.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

              precision    recall  f1-score   support

           0       0.66      0.17      0.26       253
           1       0.77      0.97      0.86       747

    accuracy                           0.77      1000
   macro avg       0.72      0.57      0.56      1000
weighted avg       0.74      0.77      0.71      1000


              precision    recall  f1-score   support

           0       0.46      0.76      0.57       253
           1       0.90      0.70      0.78       747

    accuracy                           0.71      1000
   macro avg       0.68      0.73      0.68      1000
weighted avg       0.78      0.71      0.73      1000


              precision    recall  f1-score   support

           0       0.46      0.79      0.58       253
           1       0.91      0.69      0.78       747

    accuracy                           0.72      1000
   macro avg       0.69      0.74      0.68      1000
weighted avg       0.79      0.72      0.73      1000




##Decision Tree

In [15]:
#DECISION TREE MODEL

dt = DecisionTreeClassifier()
dt.fit(xtrain, ytrain)
y_pred = dt.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

dt = DecisionTreeClassifier()
dt.fit(xtrainos, ytrainos)
y_pred = dt.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

dt = DecisionTreeClassifier()
dt.fit(xtrainus, ytrainus)
y_pred = dt.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

              precision    recall  f1-score   support

           0       0.40      0.36      0.38       253
           1       0.79      0.82      0.81       747

    accuracy                           0.70      1000
   macro avg       0.60      0.59      0.59      1000
weighted avg       0.69      0.70      0.70      1000


              precision    recall  f1-score   support

           0       0.36      0.43      0.39       253
           1       0.79      0.73      0.76       747

    accuracy                           0.66      1000
   macro avg       0.57      0.58      0.58      1000
weighted avg       0.68      0.66      0.67      1000


              precision    recall  f1-score   support

           0       0.34      0.64      0.45       253
           1       0.83      0.59      0.69       747

    accuracy                           0.60      1000
   macro avg       0.59      0.61      0.57      1000
weighted avg       0.71      0.60      0.63      1000




##Pruned Tree

In [16]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for the decision tree
param_grid = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Create a decision tree classifier
dt = DecisionTreeClassifier()

# Create the GridSearchCV object
grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, cv=5, scoring='accuracy')

# Fit the grid search to the undersampled data
grid_search.fit(xtrain, ytrain)

# Print the best parameters and best score
print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

# Evaluate the best model on the validation set
best_dt = grid_search.best_estimator_
y_pred = best_dt.predict(xvalid)
print(classification_report(yvalid, y_pred))


Best parameters: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best score: 0.73525
              precision    recall  f1-score   support

           0       0.47      0.27      0.34       253
           1       0.78      0.90      0.84       747

    accuracy                           0.74      1000
   macro avg       0.63      0.58      0.59      1000
weighted avg       0.70      0.74      0.71      1000



In [17]:
# Use the best parameters from the grid search to create oversampled and undersampled decision trees
best_params = grid_search.best_params_
print(best_params)

# Oversampled Decision Tree
dt_os = DecisionTreeClassifier(**best_params)
dt_os.fit(xtrainos, ytrainos)
y_pred_os = dt_os.predict(xvalid)
print("Oversampled Decision Tree:")
print(classification_report(yvalid, y_pred_os))

# Undersampled Decision Tree
dt_us = DecisionTreeClassifier(**best_params)
dt_us.fit(xtrainus, ytrainus)
y_pred_us = dt_us.predict(xvalid)
print("Undersampled Decision Tree:")
print(classification_report(yvalid, y_pred_us))


{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Oversampled Decision Tree:
              precision    recall  f1-score   support

           0       0.38      0.62      0.47       253
           1       0.84      0.66      0.74       747

    accuracy                           0.65      1000
   macro avg       0.61      0.64      0.60      1000
weighted avg       0.72      0.65      0.67      1000

Undersampled Decision Tree:
              precision    recall  f1-score   support

           0       0.36      0.70      0.47       253
           1       0.85      0.58      0.69       747

    accuracy                           0.61      1000
   macro avg       0.60      0.64      0.58      1000
weighted avg       0.73      0.61      0.64      1000



##Gaussian Naive Bayes

In [18]:
# prompt: use gaussian naive bayes for original, oversampled and undersampled

# Gaussian Naive Bayes Model

# ORIGINAL
gnb = GaussianNB()
gnb.fit(xtrain, ytrain)
y_pred = gnb.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

#OVERSAMPLED
gnb = GaussianNB()
gnb.fit(xtrainos, ytrainos)
y_pred = gnb.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

#UNDERSAMPLED
gnb = GaussianNB()
gnb.fit(xtrainus, ytrainus)
y_pred = gnb.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()


              precision    recall  f1-score   support

           0       0.59      0.32      0.42       253
           1       0.80      0.93      0.86       747

    accuracy                           0.77      1000
   macro avg       0.70      0.62      0.64      1000
weighted avg       0.75      0.77      0.75      1000


              precision    recall  f1-score   support

           0       0.39      0.80      0.52       253
           1       0.89      0.57      0.70       747

    accuracy                           0.63      1000
   macro avg       0.64      0.68      0.61      1000
weighted avg       0.76      0.63      0.65      1000


              precision    recall  f1-score   support

           0       0.40      0.80      0.53       253
           1       0.90      0.60      0.72       747

    accuracy                           0.65      1000
   macro avg       0.65      0.70      0.62      1000
weighted avg       0.77      0.65      0.67      1000




#KNN

In [19]:
#I scaled the variables with a 2 earlier during the read in/cleaning
#list of different neighbors
neighbors = [1, 3, 5, 7, 9, 11, 13]

for k in neighbors:
  # original
  print("ORIGINAL")
  k1 = KNeighborsClassifier(n_neighbors = k)
  k1.fit(xtrain2, ytrain)
  pred1 = k1.predict(xvalid2)
  DF1 = pd.DataFrame({"Actual": yvalid, "Predict": pred1})
  print(f"k: {k}")
  print(classification_report(yvalid, pred1))
  print()

  # Oversample
  print("OVERSAMPLE")
  k2 = KNeighborsClassifier(n_neighbors = k)
  k2.fit(xtrainos2, ytrainos)
  pred2 = k2.predict(xvalid2)
  DF2 = pd.DataFrame({"Actual": yvalid, "Predict": pred2})
  print(classification_report(yvalid, pred2))
  print()

  # undersample
  print("UNDERSAMPLE")
  k3 = KNeighborsClassifier(n_neighbors = k)
  k3.fit(xtrainus2, ytrainus)
  pred3 = k3.predict(xvalid2)
  DF3 = pd.DataFrame({"Actual": yvalid, "Predict": pred3})
  print(classification_report(yvalid, pred3))
  print()
  # Oversample K =3 appears to be the best


ORIGINAL
k: 1
              precision    recall  f1-score   support

           0       0.39      0.36      0.38       253
           1       0.79      0.81      0.80       747

    accuracy                           0.69      1000
   macro avg       0.59      0.59      0.59      1000
weighted avg       0.69      0.69      0.69      1000


OVERSAMPLE
              precision    recall  f1-score   support

           0       0.36      0.48      0.41       253
           1       0.80      0.71      0.75       747

    accuracy                           0.65      1000
   macro avg       0.58      0.59      0.58      1000
weighted avg       0.69      0.65      0.67      1000


UNDERSAMPLE
              precision    recall  f1-score   support

           0       0.33      0.64      0.44       253
           1       0.82      0.56      0.67       747

    accuracy                           0.58      1000
   macro avg       0.58      0.60      0.55      1000
weighted avg       0.70      0.58  

##Bagging

In [20]:
#Bagging
#defined the max depth and min samples based on the best params from the OS and US pruned trees above.
print("Original Bagging")
BG = BaggingClassifier(DecisionTreeClassifier(max_depth = 10, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators = 100, random_state=42)
BG.fit(xtrain, ytrain)

predBG = BG.predict(xvalid)
print(classification_report(yvalid, predBG))

print()
print("-"*60)
print("Oversampled Model")
BGOS = BaggingClassifier(DecisionTreeClassifier(max_depth = 10, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators = 100, random_state=42)
BGOS.fit(xtrainos, ytrainos)

predBGOS = BGOS.predict(xvalid)
print(classification_report(yvalid, predBGOS))

print()
print("-"*60)
print("Undersampled Model")
BGUS = BaggingClassifier(DecisionTreeClassifier(max_depth = 10, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators = 100, random_state=42)
BGUS.fit(xtrainus, ytrainus)

predBGUS = BGUS.predict(xvalid)
print(classification_report(yvalid, predBGUS))

Original Bagging


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.00      0.00      0.00       253
           1       0.75      1.00      0.86       747

    accuracy                           0.75      1000
   macro avg       0.37      0.50      0.43      1000
weighted avg       0.56      0.75      0.64      1000


------------------------------------------------------------
Oversampled Model
              precision    recall  f1-score   support

           0       0.38      0.69      0.49       253
           1       0.85      0.62      0.72       747

    accuracy                           0.64      1000
   macro avg       0.62      0.65      0.60      1000
weighted avg       0.73      0.64      0.66      1000


------------------------------------------------------------
Undersampled Model
              precision    recall  f1-score   support

           0       0.39      0.77      0.51       253
           1       0.88      0.59      0.70       747

    accuracy         

##Boosting

In [21]:
#Boosting
#defined the max depth and min samples based on the best params from the OS and US pruned trees above.
print("Original Boosting")
BT = AdaBoostClassifier(DecisionTreeClassifier(max_depth = 10, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators= 100, random_state=42)
BT.fit(xtrain, ytrain)

predBT = BT.predict(xvalid)
print(classification_report(yvalid, predBT))

print()
print("-"*60)
print("Oversampled")
BTOS = AdaBoostClassifier(DecisionTreeClassifier(max_depth = 10, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators= 100, random_state=42)
BTOS.fit(xtrainos, ytrainos)

predBTOS = BTOS.predict(xvalid)
print(classification_report(yvalid, predBTOS))

print()
print("-"*60)
print("Undersampled")
BTUS = AdaBoostClassifier(DecisionTreeClassifier(max_depth = 10, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators= 100, random_state=42)
BTUS.fit(xtrainus, ytrainus)

predBTUS = BTUS.predict(xvalid)
print(classification_report(yvalid, predBTUS))

Original Boosting
              precision    recall  f1-score   support

           0       0.57      0.13      0.21       253
           1       0.77      0.97      0.86       747

    accuracy                           0.76      1000
   macro avg       0.67      0.55      0.53      1000
weighted avg       0.72      0.76      0.69      1000


------------------------------------------------------------
Oversampled
              precision    recall  f1-score   support

           0       0.41      0.67      0.51       253
           1       0.86      0.67      0.75       747

    accuracy                           0.67      1000
   macro avg       0.63      0.67      0.63      1000
weighted avg       0.75      0.67      0.69      1000


------------------------------------------------------------
Undersampled
              precision    recall  f1-score   support

           0       0.41      0.68      0.51       253
           1       0.86      0.67      0.75       747

    accuracy   

##Random Forest

In [22]:
#Random Forest
print("Original Random Forest")
RF = RandomForestClassifier(n_estimators= 500, random_state=42)
RF.fit(xtrain, ytrain)

predRF = RF.predict(xvalid)
print(classification_report(yvalid, predRF))

print()
print("-"*60)
print("Oversampled")
RFOS = RandomForestClassifier(n_estimators= 500, random_state=42)
RFOS.fit(xtrainos, ytrainos)

predRFOS = RFOS.predict(xvalid)
print(classification_report(yvalid, predRFOS))

print()
print("-"*60)
print("Undersampled")
RFUS = RandomForestClassifier(n_estimators= 500, random_state=42)
RFUS.fit(xtrainus, ytrainus)

predRFUS = RFUS.predict(xvalid)
print(classification_report(yvalid, predRFUS))

Original Random Forest
              precision    recall  f1-score   support

           0       0.58      0.24      0.34       253
           1       0.79      0.94      0.86       747

    accuracy                           0.76      1000
   macro avg       0.68      0.59      0.60      1000
weighted avg       0.73      0.76      0.73      1000


------------------------------------------------------------
Oversampled
              precision    recall  f1-score   support

           0       0.46      0.54      0.50       253
           1       0.84      0.79      0.81       747

    accuracy                           0.72      1000
   macro avg       0.65      0.66      0.65      1000
weighted avg       0.74      0.72      0.73      1000


------------------------------------------------------------
Undersampled
              precision    recall  f1-score   support

           0       0.42      0.72      0.53       253
           1       0.87      0.66      0.75       747

    accura

##Neural Net

In [23]:
#Scaled training and test variables at the beginning during read-in/cleaning

#ORIGINAL
NN = MLPClassifier(hidden_layer_sizes = (4,5,10), activation = "relu", #try different activiations, solver, iterations, layer sizes, etc. first try activation and solvers, then layers.
                   solver = "adam", max_iter = 1000, random_state=42)
NN.fit(xtrain2, ytrain)
pred1 = NN.predict(xvalid2)
print(classification_report(yvalid, pred1))

print()

#OVERSAMPLED
NNOS = MLPClassifier(hidden_layer_sizes = (4,5,10), activation = "relu", #try different activiations, solver, iterations, layer sizes, etc. first try activation and solvers, then layers.
                   solver = "adam", max_iter = 1000, random_state=42)
NNOS.fit(xtrainos2, ytrainos)
pred1OS = NNOS.predict(xvalid2)
print(classification_report(yvalid, pred1OS))

print()

#UNDERSAMPLED
NNUS = MLPClassifier(hidden_layer_sizes = (4,5,10), activation = "relu", #try different activiations, solver, iterations, layer sizes, etc. first try activation and solvers, then layers.
                   solver = "adam", max_iter = 1000, random_state=42)
NNUS.fit(xtrainus2, ytrainus)
pred1US = NNUS.predict(xvalid2)
print(classification_report(yvalid, pred1US))

              precision    recall  f1-score   support

           0       0.70      0.23      0.35       253
           1       0.79      0.97      0.87       747

    accuracy                           0.78      1000
   macro avg       0.75      0.60      0.61      1000
weighted avg       0.77      0.78      0.74      1000


              precision    recall  f1-score   support

           0       0.41      0.85      0.55       253
           1       0.92      0.58      0.71       747

    accuracy                           0.65      1000
   macro avg       0.66      0.72      0.63      1000
weighted avg       0.79      0.65      0.67      1000


              precision    recall  f1-score   support

           0       0.39      0.83      0.53       253
           1       0.91      0.56      0.70       747

    accuracy                           0.63      1000
   macro avg       0.65      0.70      0.62      1000
weighted avg       0.78      0.63      0.65      1000



##Discriminant Analysis

In [24]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

In [25]:
# LDA
print("Original LDA")
LDA = LinearDiscriminantAnalysis()
LDA.fit(xtrain, ytrain)

predLDA = LDA.predict(xvalid)
print(classification_report(yvalid, predLDA))

print()
print("-"*60)
print("Oversampled")
LDAOS = LinearDiscriminantAnalysis()
LDAOS.fit(xtrainos, ytrainos)

predLDAOS = LDAOS.predict(xvalid)
print(classification_report(yvalid, predLDAOS))

print()
print("-"*60)
print("Undersampled")
LDAUS = LinearDiscriminantAnalysis()
LDAUS.fit(xtrainus, ytrainus)

predLDAUS = LDAUS.predict(xvalid)
print(classification_report(yvalid, predLDAUS))

print()
print("="*60)

Original LDA
              precision    recall  f1-score   support

           0       0.71      0.16      0.26       253
           1       0.77      0.98      0.86       747

    accuracy                           0.77      1000
   macro avg       0.74      0.57      0.56      1000
weighted avg       0.76      0.77      0.71      1000


------------------------------------------------------------
Oversampled
              precision    recall  f1-score   support

           0       0.45      0.77      0.57       253
           1       0.90      0.68      0.77       747

    accuracy                           0.70      1000
   macro avg       0.67      0.72      0.67      1000
weighted avg       0.78      0.70      0.72      1000


------------------------------------------------------------
Undersampled
              precision    recall  f1-score   support

           0       0.45      0.79      0.58       253
           1       0.91      0.67      0.77       747

    accuracy        

##Support Vector Machine

In [26]:
from sklearn.svm import SVC

print("Original SVM")
SVM = SVC(kernel='linear')  # Only tried with linear for sake of time, but coud have tried with rbf,  poly, or sigmoid
SVM.fit(xtrain, ytrain)

predSVM = SVM.predict(xvalid)
print(classification_report(yvalid, predSVM))

print()
print("-"*60)
print("Oversampled")
SVMOS = SVC(kernel='linear')
SVMOS.fit(xtrainos, ytrainos)

predSVMOS = SVMOS.predict(xvalid)
print(classification_report(yvalid, predSVMOS))

print()
print("-"*60)
print("Undersampled")
SVMUS = SVC(kernel='linear')
SVMUS.fit(xtrainus, ytrainus)

predSVMUS = SVMUS.predict(xvalid)
print(classification_report(yvalid, predSVMUS))

print()
print("="*60)


Original SVM


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.00      0.00      0.00       253
           1       0.75      1.00      0.86       747

    accuracy                           0.75      1000
   macro avg       0.37      0.50      0.43      1000
weighted avg       0.56      0.75      0.64      1000


------------------------------------------------------------
Oversampled
              precision    recall  f1-score   support

           0       0.44      0.79      0.57       253
           1       0.90      0.66      0.76       747

    accuracy                           0.69      1000
   macro avg       0.67      0.73      0.66      1000
weighted avg       0.79      0.69      0.71      1000


------------------------------------------------------------
Undersampled
              precision    recall  f1-score   support

           0       0.44      0.79      0.57       253
           1       0.90      0.66      0.76       747

    accuracy                     

#Part 2

In [27]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.neighbors import KNeighborsClassifier

In [28]:
glass = pd.read_csv("glass.csv")
print(glass.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RI      214 non-null    float64
 1   Na      214 non-null    float64
 2   Mg      214 non-null    float64
 3   Al      214 non-null    float64
 4   Si      214 non-null    float64
 5   K       214 non-null    float64
 6   Ca      214 non-null    float64
 7   Ba      214 non-null    float64
 8   Fe      214 non-null    float64
 9   Type    214 non-null    int64  
dtypes: float64(9), int64(1)
memory usage: 16.8 KB
None


In [29]:
glass.head()

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [30]:
#Determine beta coefficients and p-values
inputs = 'RI + Na + Mg + Al + Si + K + Ca + Ba + Fe'
lm = smf.ols(formula = f'Type ~ {inputs}', data = glass).fit()

In [31]:
lm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   Type   R-squared:                       0.747
Model:                            OLS   Adj. R-squared:                  0.735
Method:                 Least Squares   F-statistic:                     66.75
Date:                Thu, 26 Mar 2026   Prob (F-statistic):           5.77e-56
Time:                        01:40:17   Log-Likelihood:                -315.46
No. Observations:                 214   AIC:                             650.9
Df Residuals:                     204   BIC:                             684.6
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -204.2071    131.115     -1.557      0.121    -462.722      54.308
RI            87.0012     75.283      1.156      0.249     -61.431     235.433
Na             1.1047      0.714      1.548      0.123      -0.303       2.512
Mg            -0.3230      0.741     -0.436      0.663      -1.784       1.138
Al             1.5923      0.753      2.115      0.036       0.108       3.076
Si             0.7768      0.730      1.064      0.289      -0.663       2.217
K              0.4108      0.751      0.547      0.585      -1.070       1.892
Ca             0.2122      0.758      0.280      0.780      -1.282       1.706
Ba             0.8199      0.762      1.075      0.284      -0.683       2.323
Fe            -0.7529      0.840     -0.896      0.371      -2.409       0.903
==============================================================================
Omnibus:                       48.035   Durbin-Watson:                   0.689
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               89.096
Skew:                           1.128   Prob(JB):                     4.50e-20
Kurtosis:                       5.213   Cond. No.                     1.48e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.48e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [32]:
#drop unncessary variables
glass = glass.drop(columns = ["RI"])

In [33]:
#deterimine predictors(X) and outcome(y)
predictors = ['Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe']

In [34]:
y = glass['Type'].astype('category')
print(y.info())

<class 'pandas.core.series.Series'>
RangeIndex: 214 entries, 0 to 213
Series name: Type
Non-Null Count  Dtype   
--------------  -----   
214 non-null    category
dtypes: category(1)
memory usage: 566.0 bytes
None


In [35]:
glass["Type"].value_counts()

,count
Type,
2,76
1,70
7,29
3,17
5,13
6,9


In [36]:
#handle categorical variables
X = pd.get_dummies(glass[predictors], drop_first= True)
#didn't really need to define it this way since all are float64 already.

In [37]:
print(X.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Na      214 non-null    float64
 1   Mg      214 non-null    float64
 2   Al      214 non-null    float64
 3   Si      214 non-null    float64
 4   K       214 non-null    float64
 5   Ca      214 non-null    float64
 6   Ba      214 non-null    float64
 7   Fe      214 non-null    float64
dtypes: float64(8)
memory usage: 13.5 KB
None


In [38]:
xtrain, xvalid, ytrain, yvalid = train_test_split(X, y,
                                                      test_size = 0.2, random_state=42)

print(xtrain.info())
print()
print(ytrain.info())
print()
print(xvalid.info())
print()
print(yvalid.info())

<class 'pandas.core.frame.DataFrame'>
Index: 171 entries, 79 to 102
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Na      171 non-null    float64
 1   Mg      171 non-null    float64
 2   Al      171 non-null    float64
 3   Si      171 non-null    float64
 4   K       171 non-null    float64
 5   Ca      171 non-null    float64
 6   Ba      171 non-null    float64
 7   Fe      171 non-null    float64
dtypes: float64(8)
memory usage: 12.0 KB
None

<class 'pandas.core.series.Series'>
Index: 171 entries, 79 to 102
Series name: Type
Non-Null Count  Dtype   
--------------  -----   
171 non-null    category
dtypes: category(1)
memory usage: 1.7 KB
None

<class 'pandas.core.frame.DataFrame'>
Index: 43 entries, 9 to 60
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Na      43 non-null     float64
 1   Mg      43 non-null     float64
 2   Al      43 non-null     flo

In [39]:
overs = SMOTE()
unders = RandomUnderSampler() # don't use sampling_strategy = majority for more than 2 categories

xtrainos, ytrainos = overs.fit_resample(xtrain, ytrain)
xtrainus, ytrainus = unders.fit_resample(xtrain, ytrain)

print(ytrainos.value_counts())
print(ytrainus.value_counts())

Type
1    62
2    62
3    62
5    62
6    62
7    62
Name: count, dtype: int64
Type
1    6
2    6
3    6
5    6
6    6
7    6
Name: count, dtype: int64


In [40]:
#Normalizing the dataset

scale = StandardScaler()
xtrain2 = pd.DataFrame(scale.fit_transform(xtrain), columns = X.columns)
xtrainos2 = pd.DataFrame(scale.fit_transform(xtrainos), columns = X.columns)
xtrainus2 = pd.DataFrame(scale.fit_transform(xtrainus), columns = X.columns)
xvalid2 = pd.DataFrame(scale.fit_transform(xvalid), columns = X.columns)

##KNN

In [41]:
#list of different neighbors
neighbors = [1, 3, 5, 7, 9]

for k in neighbors:
  # original
  print("ORIGINAL")
  k1 = KNeighborsClassifier(n_neighbors = k)
  k1.fit(xtrain2, ytrain)
  pred1 = k1.predict(xvalid2)
  DF1 = pd.DataFrame({"Actual": yvalid, "Predict": pred1})
  print(f"k: {k}")
  print(classification_report(yvalid, pred1))
  print()

  # Oversample
  print("OVERSAMPLE")
  k2 = KNeighborsClassifier(n_neighbors = k)
  k2.fit(xtrainos2, ytrainos)
  pred2 = k2.predict(xvalid2)
  DF2 = pd.DataFrame({"Actual": yvalid, "Predict": pred2})
  print(classification_report(yvalid, pred2))
  print()

  # undersample
  print("UNDERSAMPLE")
  k3 = KNeighborsClassifier(n_neighbors = k)
  k3.fit(xtrainus2, ytrainus)
  pred3 = k3.predict(xvalid2)
  DF3 = pd.DataFrame({"Actual": yvalid, "Predict": pred3})
  print(classification_report(yvalid, pred3))
  print()
  # Oversample K=9 appears to be the best


ORIGINAL
k: 1
              precision    recall  f1-score   support

           1       0.53      0.73      0.62        11
           2       0.64      0.64      0.64        14
           3       0.50      0.33      0.40         3
           5       1.00      0.75      0.86         4
           6       1.00      0.67      0.80         3
           7       1.00      0.88      0.93         8

    accuracy                           0.70        43
   macro avg       0.78      0.67      0.71        43
weighted avg       0.73      0.70      0.70        43


OVERSAMPLE
              precision    recall  f1-score   support

           1       0.62      0.73      0.67        11
           2       0.82      0.64      0.72        14
           3       0.50      0.67      0.57         3
           5       1.00      1.00      1.00         4
           6       1.00      1.00      1.00         3
           7       1.00      1.00      1.00         8

    accuracy                           0.79        

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

##Categorical Trees

In [42]:
#DECISION TREE MODEL

dt = DecisionTreeClassifier()
dt.fit(xtrain, ytrain)
y_pred = dt.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

dt = DecisionTreeClassifier()
dt.fit(xtrainos, ytrainos)
y_pred = dt.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

dt = DecisionTreeClassifier()
dt.fit(xtrainus, ytrainus)
y_pred = dt.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

              precision    recall  f1-score   support

           1       0.56      0.82      0.67        11
           2       0.75      0.64      0.69        14
           3       0.50      0.33      0.40         3
           5       1.00      0.25      0.40         4
           6       0.75      1.00      0.86         3
           7       1.00      1.00      1.00         8

    accuracy                           0.72        43
   macro avg       0.76      0.67      0.67        43
weighted avg       0.75      0.72      0.71        43


              precision    recall  f1-score   support

           1       0.73      0.73      0.73        11
           2       0.90      0.64      0.75        14
           3       0.40      0.67      0.50         3
           5       0.67      1.00      0.80         4
           6       1.00      1.00      1.00         3
           7       1.00      1.00      1.00         8

    accuracy                           0.79        43
   macro avg       0.7

##Pruned Tree

In [43]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for the decision tree
param_grid = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Create a decision tree classifier
dt = DecisionTreeClassifier()

# Create the GridSearchCV object
grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, cv=5, scoring='accuracy')

# Fit the grid search to the undersampled data
grid_search.fit(xtrain, ytrain)

# Print the best parameters and best score
print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

# Evaluate the best model on the validation set
best_dt = grid_search.best_estimator_
y_pred = best_dt.predict(xvalid)
print(classification_report(yvalid, y_pred))

Best parameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5}
Best score: 0.6315966386554621
              precision    recall  f1-score   support

           1       0.53      0.82      0.64        11
           2       0.86      0.43      0.57        14
           3       0.25      0.33      0.29         3
           5       1.00      0.25      0.40         4
           6       0.50      1.00      0.67         3
           7       1.00      1.00      1.00         8

    accuracy                           0.65        43
   macro avg       0.69      0.64      0.59        43
weighted avg       0.75      0.65      0.64        43



In [44]:
# Use the best parameters from the grid search to create oversampled and undersampled decision trees
best_params = grid_search.best_params_
print(best_params)

# Oversampled Decision Tree
dt_os = DecisionTreeClassifier(**best_params)
dt_os.fit(xtrainos, ytrainos)
y_pred_os = dt_os.predict(xvalid)
print("Oversampled Decision Tree:")
print(classification_report(yvalid, y_pred_os))

# Undersampled Decision Tree
dt_us = DecisionTreeClassifier(**best_params)
dt_us.fit(xtrainus, ytrainus)
y_pred_us = dt_us.predict(xvalid)
print("Undersampled Decision Tree:")
print(classification_report(yvalid, y_pred_us))


{'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5}
Oversampled Decision Tree:
              precision    recall  f1-score   support

           1       0.64      0.64      0.64        11
           2       0.89      0.57      0.70        14
           3       0.33      0.67      0.44         3
           5       0.67      1.00      0.80         4
           6       1.00      0.67      0.80         3
           7       0.89      1.00      0.94         8

    accuracy                           0.72        43
   macro avg       0.74      0.76      0.72        43
weighted avg       0.77      0.72      0.73        43

Undersampled Decision Tree:
              precision    recall  f1-score   support

           1       0.47      0.64      0.54        11
           2       0.50      0.36      0.42        14
           3       0.50      0.67      0.57         3
           5       0.00      0.00      0.00         4
           6       0.75      1.00      0.86         3
          

##Gaussian Naive Bayes

In [45]:
# prompt: use gaussian naive bayes for original, oversampled and undersampled

# Gaussian Naive Bayes Model

# ORIGINAL
gnb = GaussianNB()
gnb.fit(xtrain, ytrain)
y_pred = gnb.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

#OVERSAMPLED
gnb = GaussianNB()
gnb.fit(xtrainos, ytrainos)
y_pred = gnb.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()

#UNDERSAMPLED
gnb = GaussianNB()
gnb.fit(xtrainus, ytrainus)
y_pred = gnb.predict(xvalid)
print(classification_report(yvalid, y_pred))
print()


              precision    recall  f1-score   support

           1       0.39      0.64      0.48        11
           2       0.43      0.21      0.29        14
           3       0.25      0.33      0.29         3
           5       0.50      0.25      0.33         4
           6       1.00      1.00      1.00         3
           7       0.89      1.00      0.94         8

    accuracy                           0.53        43
   macro avg       0.58      0.57      0.55        43
weighted avg       0.54      0.53      0.51        43


              precision    recall  f1-score   support

           1       0.40      0.18      0.25        11
           2       0.60      0.21      0.32        14
           3       0.18      1.00      0.30         3
           5       0.75      0.75      0.75         4
           6       1.00      0.67      0.80         3
           7       0.80      1.00      0.89         8

    accuracy                           0.49        43
   macro avg       0.6

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


##Bagging

In [46]:
#Bagging
#defined the max depth and min samples based on the best params from the OS and US pruned trees above.
print("Original Bagging")
BG = BaggingClassifier(DecisionTreeClassifier(max_depth = 30, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators = 100, random_state=42)
BG.fit(xtrain, ytrain)

predBG = BG.predict(xvalid)
print(classification_report(yvalid, predBG))

print()
print("-"*60)
print("Oversampled Model")
BGOS = BaggingClassifier(DecisionTreeClassifier(max_depth = 30, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators = 100, random_state=42)
BGOS.fit(xtrainos, ytrainos)

predBGOS = BGOS.predict(xvalid)
print(classification_report(yvalid, predBGOS))

print()
print("-"*60)
print("Undersampled Model")
BGUS = BaggingClassifier(DecisionTreeClassifier(max_depth = 30, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators = 100, random_state=42)
BGUS.fit(xtrainus, ytrainus)

predBGUS = BGUS.predict(xvalid)
print(classification_report(yvalid, predBGUS))

Original Bagging


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           1       0.67      0.91      0.77        11
           2       0.83      0.71      0.77        14
           3       0.00      0.00      0.00         3
           5       0.80      1.00      0.89         4
           6       1.00      1.00      1.00         3
           7       1.00      1.00      1.00         8

    accuracy                           0.81        43
   macro avg       0.72      0.77      0.74        43
weighted avg       0.77      0.81      0.79        43


------------------------------------------------------------
Oversampled Model
              precision    recall  f1-score   support

           1       0.57      0.73      0.64        11
           2       1.00      0.57      0.73        14
           3       0.50      0.67      0.57         3
           5       0.67      1.00      0.80         4
           6       1.00      1.00      1.00         3
           7       1.00      1.00      1.00         

##Boosting

In [47]:
#Boosting
#defined the max depth and min samples based on the best params from the OS and US pruned trees above.
print("Original Boosting")
BT = AdaBoostClassifier(DecisionTreeClassifier(max_depth = 30, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators= 100, random_state=42)
BT.fit(xtrain, ytrain)

predBT = BT.predict(xvalid)
print(classification_report(yvalid, predBT))

print()
print("-"*60)
print("Oversampled")
BTOS = AdaBoostClassifier(DecisionTreeClassifier(max_depth = 30, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators= 100, random_state=42)
BTOS.fit(xtrainos, ytrainos)

predBTOS = BTOS.predict(xvalid)
print(classification_report(yvalid, predBTOS))

print()
print("-"*60)
print("Undersampled")
BTUS = AdaBoostClassifier(DecisionTreeClassifier(max_depth = 30, min_samples_split = 5,
                                 min_impurity_decrease = 0.01), n_estimators= 100, random_state=42)
BTUS.fit(xtrainus, ytrainus)

predBTUS = BTUS.predict(xvalid)
print(classification_report(yvalid, predBTUS))

Original Boosting
              precision    recall  f1-score   support

           1       0.71      0.91      0.80        11
           2       0.77      0.71      0.74        14
           3       1.00      0.33      0.50         3
           5       0.67      0.50      0.57         4
           6       1.00      1.00      1.00         3
           7       0.89      1.00      0.94         8

    accuracy                           0.79        43
   macro avg       0.84      0.74      0.76        43
weighted avg       0.80      0.79      0.78        43


------------------------------------------------------------
Oversampled
              precision    recall  f1-score   support

           1       0.67      0.73      0.70        11
           2       0.82      0.64      0.72        14
           3       0.50      0.67      0.57         3
           5       0.75      0.75      0.75         4
           6       1.00      1.00      1.00         3
           7       0.89      1.00      0

##Multinomial Logistic Regression

In [48]:
# Multinomial Logistic Regression
print("Original Multinomial Logistic Regression")
MLR = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=500, random_state=42)
MLR.fit(xtrain, ytrain)

predMLR = MLR.predict(xvalid)
print(classification_report(yvalid, predMLR))

print()
print("-"*60)
print("Oversampled")
MLROS = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=500, random_state=42)
MLROS.fit(xtrainos, ytrainos)

predMLROS = MLROS.predict(xvalid)
print(classification_report(yvalid, predMLROS))

print()
print("-"*60)
print("Undersampled")
MLRUS = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=500, random_state=42)
MLRUS.fit(xtrainus, ytrainus)

predMLRUS = MLRUS.predict(xvalid)
print(classification_report(yvalid, predMLRUS))

print()
print("="*60)


Original Multinomial Logistic Regression
              precision    recall  f1-score   support

           1       0.75      0.82      0.78        11
           2       0.60      0.86      0.71        14
           3       0.00      0.00      0.00         3
           5       1.00      0.50      0.67         4
           6       0.00      0.00      0.00         3
           7       0.78      0.88      0.82         8

    accuracy                           0.70        43
   macro avg       0.52      0.51      0.50        43
weighted avg       0.62      0.70      0.65        43


------------------------------------------------------------
Oversampled


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisi

              precision    recall  f1-score   support

           1       0.73      0.73      0.73        11
           2       0.83      0.36      0.50        14
           3       0.22      0.67      0.33         3
           5       0.80      1.00      0.89         4
           6       1.00      1.00      1.00         3
           7       0.89      1.00      0.94         8

    accuracy                           0.70        43
   macro avg       0.75      0.79      0.73        43
weighted avg       0.78      0.70      0.70        43


------------------------------------------------------------
Undersampled
              precision    recall  f1-score   support

           1       0.50      0.18      0.27        11
           2       1.00      0.07      0.13        14
           3       0.10      0.67      0.17         3
           5       0.57      1.00      0.73         4
           6       0.75      1.00      0.86         3
           7       1.00      0.88      0.93         8

  

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of